# Data Preprocessing

In [1]:
import pandas as pd
import os
import os.path as osp
import csv

In [2]:
log_path = "data/data.txt"
name_list = ["reno", "tri", "mario", "nopal"]
min_len_columns = 31 # pake minimum karena ada tim tri totalnya ga sama, harusnya 32

In [3]:
def write_column_names_to_csv(column_names, csv_path):
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(column_names)
        
def write_line_to_csv(line, csv_path):
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(line)

In [4]:
with open(log_path, "r", encoding="utf-16", errors="ignore") as file:
    name_counter = 0
    is_numerical_data = False
    is_real_flight = False
    column_names = list()

    for line in file:
        lines = [ln.strip().replace("|", "") for ln in line.split("|")]
        lines.pop()

        if len(lines) == 0:
            continue

        # check if it's the actual flight data not empty data yang dikosongin mufty
        # "f-act,_/sec" condition because for some reason tim nopal ada column itu
        if len(lines) >= min_len_columns and 'f-act,_/sec' not in lines:
            is_real_flight = True
        else:
            is_real_flight = False
            # continue
        
        # check if there's a numerical data in the line, because if it's not, it's a column name
        try:
            is_numerical_data = True if float(lines[0]) else False
        except ValueError:
            is_numerical_data = False
        
        if len(column_names) == 0 and is_real_flight:
            column_names = [ln.replace("_","") for ln in lines]
            print(column_names)
        
        # condition when the real flight data has ended and the next lines are the empty data from mufty, > 0 is to avoid the first set condition
        if len(column_names) > 0 and not is_numerical_data and not is_real_flight:
            name_counter += 1
            print(f"Finished processing {name_list[name_counter-1]}. Moving to {name_list[name_counter]}...")
            column_names = list()

        if is_numerical_data and is_real_flight and len(column_names) > 0:
            csv_path = f"data/output_{name_list[name_counter]}.csv"
            if not osp.exists(csv_path):
                os.makedirs(osp.dirname(csv_path), exist_ok=True)
                write_line_to_csv(column_names, csv_path)
            write_line_to_csv(lines, csv_path)
        
        print(f"is_numerical_data: {is_numerical_data}, is_real_flight: {is_real_flight}, lines: {lines}")


is_numerical_data: False, is_real_flight: False, lines: ['_real,_time', '_totl,_time', 'missn,_time', 'timer,_time', '_zulu,_time', 'local,_time', 'hobbs,_time']
is_numerical_data: False, is_real_flight: False, lines: ['_real,_time', '_totl,_time', 'missn,_time', 'timer,_time', '_zulu,_time', 'local,_time', 'hobbs,_time']
is_numerical_data: False, is_real_flight: False, lines: ['_real,_time', '_totl,_time', 'missn,_time', 'timer,_time', '_zulu,_time', 'local,_time', 'hobbs,_time', '_Vind,_kias', '_Vind,_keas', 'Vtrue,_ktas', 'Vtrue,_ktgs', '_Vind,__mph', 'Vtrue,mphas', 'Vtrue,mphgs']
is_numerical_data: True, is_real_flight: False, lines: ['547.43866', '259.55338', '259.55338', '0.00000', '5.09210', '12.09210', '24.59312', '0.00000', '0.00974', '0.00976', '0.00975', '0.00000', '0.01123', '0.01123']
is_numerical_data: True, is_real_flight: False, lines: ['547.55133', '259.69339', '259.69339', '0.00000', '5.09214', '12.09214', '24.59316', '0.00000', '0.00863', '0.00865', '0.00864', '0.000

# Analysis

In [5]:
import pandas as pd

In [6]:
data = pd.read_csv("data/output_mario.csv")
data.head()

,"real,time","totl,time","missn,time","timer,time","zulu,time","local,time","hobbs,time","Vind,kias","Vind,keas","Vtrue,ktas",...,"hding,mag","lat,deg","lon,deg","alt,ftmsl","alt,ftagl","on,runwy","alt,ind","lat,orign","lon,orign","thro1,part"
0,1977.02808,1174.88281,0.00364,0.0,5.34636,12.34636,0.00256,0.00000,0.15184,0.15208,...,64.71442,-6.27214,106.87902,88.26725,0.00114,1.0,0.00825,-6.5,107.0,1.0
1,1977.12744,1174.98218,0.10309,0.0,5.34638,12.34638,0.00258,0.00000,1.06911,1.07081,...,64.71388,-6.27214,106.87902,88.40345,0.13735,1.0,1.74388,-6.5,107.0,1.0
2,1977.22717,1175.08191,0.20276,0.0,5.34641,12.34641,0.00261,0.00000,0.66836,0.66942,...,64.71378,-6.27214,106.87902,88.55391,0.28791,1.0,5.74815,-6.5,107.0,1.0
3,1977.32666,1175.18140,0.30224,0.0,5.34644,12.34644,0.00264,0.00000,0.34537,0.34592,...,64.71409,-6.27214,106.87902,88.60777,0.34193,1.0,11.13593,-6.5,107.0,1.0
4,1977.42517,1175.27991,0.40080,0.0,5.34647,12.34647,0.00267,0.00636,0.42415,0.42482,...,64.71486,-6.27214,106.87902,88.60377,0.33817,1.0,17.22369,-6.5,107.0,1.0
